In [86]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import plotly.express as px
# Import the processing module from the same folder
from processing import load_solutions, add_kwargs_as_indices, combine_solutions, read_parquet_and_convert, add_fields
# import processing 
# from pivottablejs import pivot_ui
G_save = True

In [87]:
def create_envelope(s_ed, s_uc, group_by = ['configuration', 'µ', 'iteration', 'day', 'hour,', 'r_id']):
    # Copy the relevant columns from s_ed['storage']
    envelope = s_ed['storage'][group_by + ['SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].copy()

    # Perform the first left join
    envelope = envelope.merge(
        s_uc['storage'][[col for col in group_by if col != 'iteration'] + ['SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].rename(
            columns={'SOE_MWh': 'SOE_DA_MWh', 'envelope_up_MWh': 'envelope_up_DA_MWh', 'envelope_down_MWh': 'envelope_down_DA_MWh'}
        ),
        on=[col for col in group_by if col != 'iteration'],
        how='left'
    )
    
    # Perform the second left join
    envelope = envelope.merge(
        s_uc['storage_parameters'][['r_id', 'SOE_max_MWh', 'initial_energy_proportion']].drop_duplicates(),
        on='r_id',
        how='left'
    )
    
    # Calculate initial state of energy (SOE) based on maximum SOE and initial energy proportion
    envelope['SOE_0_MWh'] = envelope['SOE_max_MWh'] * envelope['initial_energy_proportion']
    envelope['SOC'] = envelope['SOE_MWh'] / envelope['SOE_max_MWh'] 
    # Group by day, configuration, and resource ID, and get the last entry for each group

    return envelope

In [ ]:

ss = [
    # {'solution_folder': f"RTS-GMLC_v5.2.2s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2z", 'model_type' : 'envelope'}, # last opertional
    # {'solution_folder': f"RTS-GMLC_v5.2.2.1s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.3z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.3.1z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.4z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.4.1z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v6.2.2s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v15.0s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v16.0.1su", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v16.0.2su", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v16.1su", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v16.2su", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v9.0su", 'VLGEN': 30, 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v10.0s", 'VLGEN': 30, 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v6.2.3s", 'model_type' : 'e-reserve'}
    ]

s_uc = []
s_ed = []
gcd_KPI_adequacy = []
gcdi_KPI_adequacy = []
for sol in ss:
    # ρ = sol['ρ']
    s = sol['solution_folder']
    s_uc_name = 's_suc' if sol['model_type'] == 'stochastic' else 's_uc'
    # s_uc_ = load_solutions(s_uc_name, os.path.join("..", "output", s), [7], model_type = sol['model_type'], ρ=ρ, solution_id = s)
    # s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), [7], model_type = sol['model_type'], ρ=ρ, solution_id = s)
    # s_uc.append(s_uc_)
    # s_ed.append(s_ed_)

    gcd_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcd_KPI_adequacy.parquet"))
    gcd_KPI_adequacy_ = add_fields(gcd_KPI_adequacy_, model_type = sol['model_type'], solution_id = s) 

    gcdi_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcdi_KPI_adequacy.parquet"))
    gcdi_KPI_adequacy_ = add_fields(gcdi_KPI_adequacy_, model_type = sol['model_type'], solution_id = s)

    gcd_KPI_adequacy.append(gcd_KPI_adequacy_)
    gcdi_KPI_adequacy.append(gcdi_KPI_adequacy_)

# s_uc = combine_solutions(s_uc)
# s_ed = combine_solutions(s_ed)
gcd_KPI_adequacy = pd.concat(gcd_KPI_adequacy)
gcdi_KPI_adequacy = pd.concat(gcdi_KPI_adequacy)

if 'µ' in gcdi_KPI_adequacy.columns: 
#         # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
    gcdi_KPI_adequacy['model_type'] = gcdi_KPI_adequacy.apply(lambda x: 'conservative' if ('envelope' in x['model_type']) & (x['µ'] == 1) else x['model_type'], axis=1)
    gcd_KPI_adequacy['model_type'] = gcd_KPI_adequacy.apply(lambda x: 'conservative' if ('envelope' in x['model_type']) & (x['µ'] == 1) else x['model_type'], axis=1)




../output/RTS-GMLC_v16.0.1su/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_v16.0.1su/all_gcdi_KPI_adequacy.parquet
../output/RTS-GMLC_v16.1su/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_v16.1su/all_gcdi_KPI_adequacy.parquet
../output/RTS-GMLC_v16.2su/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_v16.2su/all_gcdi_KPI_adequacy.parquet


In [89]:
gcdi_KPI_adequacy['model_type'].unique()

array(['conservative', 'envelope'], dtype=object)

In [90]:
if G_save:
    out= gcd_KPI_adequacy.copy()
    if 'µ' in out.columns: 
        out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
        # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: ((x[0] =='envelope')*(x[1]==1)*'conservatice' + x[0]), axis = 1)

    renames = {'µ': 'mu', 'ρ' : 'rho'}
    renames = {k: v for k, v in renames.items() if k in out.columns}
    out.rename(columns = renames, inplace = True)
    out.to_csv('gcd_KPI_adequacy.csv', index=False)
    gcdi_KPI_adequacy.rename(columns = renames, inplace = True)
    gcdi_KPI_adequacy.reset_index().to_csv('gcdi_KPI_adequacy.csv', index=False)

/tmp/ipykernel_7545/3797213802.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)


In [91]:
gcdi_KPI_adequacy

,model_type,solution_id,configuration,day,iteration,LLD_h,ENS_MWh,input_load_MWh,CURD_h,CUR_MWh,...,required_reserve_down_uc_MWh,slack_reserve_down_uc_MWh,thermal_reserve_up_uc_MWh,thermal_reserve_down_uc_MWh,storage_reserve_up_uc_MWh,storage_reserve_down_uc_MWh,thermal_reserve_up_activation_MWh,thermal_reserve_down_activation_MWh,storage_reserve_up_activation_MWh,storage_reserve_down_activation_MWh
0,conservative,RTS-GMLC_v16.0.1su,base_ramp_storage_envelopes_up_1_dn_1,2,demand_1,0,2.655625e-13,43036.274891,0,7.638334e-14,...,15674.750637,1.238774e-10,5319.604141,8349.747125,11395.244324,7325.543322,0.000000,8834.638765,8834.638766,0.000000
1,envelope,RTS-GMLC_v16.0.1su,base_ramp_storage_envelopes_up_0_5_dn_0_5,2,demand_1,0,8.263007e-15,43036.274891,0,7.638334e-14,...,15674.750637,7.238277e-12,3183.660746,9145.691824,13530.980058,6529.070181,7.887143,5292.942351,5292.942351,7.887143
2,conservative,RTS-GMLC_v16.0.1su,base_ramp_storage_envelopes_up_1_dn_1,3,demand_1,0,1.123630e-15,41727.386042,0,-6.572520e-14,...,20866.187897,2.144523e-11,10511.731690,8614.895199,12054.084814,12252.547119,0.000000,10600.499356,10600.499357,0.000000
3,envelope,RTS-GMLC_v16.0.1su,base_ramp_storage_envelopes_up_0_5_dn_0_5,3,demand_1,0,2.723901e-12,41727.386042,0,-6.572520e-14,...,20866.187897,6.437350e-11,4712.893906,7776.985022,17852.899101,13089.313160,0.000000,7004.849100,7004.849101,0.000000
0,conservative,RTS-GMLC_v16.1su,base_ramp_storage_envelopes_up_1_dn_1,2,demand_1,0,2.100746e-17,42264.410558,0,7.638334e-14,...,15674.750637,1.238774e-10,5319.604141,8349.747125,11395.244324,7325.543322,963.180278,1780.892397,2737.905268,2692.057480
1,envelope,RTS-GMLC_v16.1su,base_ramp_storage_envelopes_up_0_5_dn_0_5,2,demand_1,0,2.023607e-14,42264.410558,0,7.638334e-14,...,15674.750637,7.238277e-12,3183.660746,9145.691824,13530.980058,6529.070181,1040.578379,1877.757712,2624.397948,2559.082947
2,conservative,RTS-GMLC_v16.1su,base_ramp_storage_envelopes_up_1_dn_1,3,demand_1,0,9.986932e-12,39266.476180,0,-6.572520e-14,...,20866.187897,2.144523e-11,10511.731690,8614.895199,12054.084814,12252.547119,841.608227,3173.090210,3821.350827,3950.778706
3,envelope,RTS-GMLC_v16.1su,base_ramp_storage_envelopes_up_0_5_dn_0_5,3,demand_1,0,9.421030e-16,39266.476180,0,-6.572520e-14,...,20866.187897,6.437350e-11,4712.893906,7776.985022,17852.899101,13089.313160,325.678780,2860.744904,4228.743287,4154.587025
0,conservative,RTS-GMLC_v16.2su,base_ramp_storage_envelopes_up_1_dn_1,2,demand_1,0,1.864005e-13,42264.410558,0,7.638334e-14,...,15674.750637,1.238774e-10,5319.604141,8349.747125,11395.244324,7325.543322,0.000000,8834.638765,8498.783118,436.008685
1,envelope,RTS-GMLC_v16.2su,base_ramp_storage_envelopes_up_0_5_dn_0_5,2,demand_1,0,8.113627e-12,42264.410558,0,7.638334e-14,...,15674.750637,7.238277e-12,3183.660746,9145.691824,13530.980058,6529.070181,106.385551,6134.088266,6795.499058,1539.660674
